# Dataset Final — Features de course (par année)
Combine : matching + Komoot surfaces + features GPX
Output : dataset_final_{year}.csv par année, puis dataset_final_all.csv fusionné

In [13]:
import os
import re
import numpy as np
import pandas as pd
import gpxpy
from scipy.signal import find_peaks
from tqdm import tqdm

BASE       = '/Users/arthurdeletang/Desktop/Stage M1/Code/data'
GPX_DIR    = os.path.join(BASE, 'gpx_files_2')
MATCH_DIR  = os.path.join(BASE, 'matching', 'all')
KOMOOT_CSV = os.path.join(MATCH_DIR, 'komoot_surface_all.csv')
OUTPUT_DIR = os.path.join(BASE, 'matching', 'by_year')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir : {OUTPUT_DIR}')

Output dir : /Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/by_year


In [14]:
import re

def parse_gpx_safe(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    content = re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#)', '&amp;', content)
    return gpxpy.parse(content)

def find_gpx_path(fname, gpx_dir):
    full_path = os.path.join(gpx_dir, fname)
    if os.path.exists(full_path):
        return full_path
    truncated = fname.split(' / ')[0].strip() + '.gpx'
    trunc_path = os.path.join(gpx_dir, truncated)
    if os.path.exists(trunc_path):
        return trunc_path
    return None

def get_number_peaks(series, prominence):
    store = find_peaks(series, prominence=prominence)
    try:
        if (len(store[0]) > 1) and (np.min(np.diff(store[0])) < prominence):
            return len(store[1]['prominences'][np.diff(np.insert(store[0], 0, 0)) > prominence])
        else:
            return len(store[1]['prominences'])
    except:
        return len(store[1]['prominences'])

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def gradient_last_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    for i in range(len(lats)-1, 0, -1):
        dist_cumul += haversine(lats[i-1], lons[i-1], lats[i], lons[i])
        if dist_cumul >= km:
            elev_start = elevations[i-1]
            elev_end   = elevations[-1]
            deniv = elev_end - elev_start
            return round(deniv / (km * 1000) * 100, 2)
    total_dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1])
                     for i in range(len(lats)-1))
    if total_dist == 0:
        return 0.0
    return round((elevations[-1] - elevations[0]) / (total_dist * 1000) * 100, 2)

def gradient_first_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        if dist_cumul >= km:
            deniv = elevations[i+1] - elevations[0]
            return round(deniv / (km * 1000) * 100, 2)
    total_dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
    if total_dist == 0:
        return 0.0
    return round((elevations[-1] - elevations[0]) / (total_dist * 1000) * 100, 2)

def denivele_first_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    deniv = 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        diff = elevations[i+1] - elevations[i]
        if diff > 0:
            deniv += diff
        if dist_cumul >= km:
            break
    return round(deniv, 0)

def extract_gpx_features(filepath):
    try:
        gpx = parse_gpx_safe(filepath)
        points = gpx.tracks[0].segments[0].points
        elevations = [p.elevation or 0 for p in points]
        lats = [p.latitude for p in points]
        lons = [p.longitude for p in points]

        dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1])
                   for i in range(len(lats)-1))

        elev_diff = np.diff(elevations)
        denivele_pos = float(np.sum(elev_diff[elev_diff > 0]))
        denivele_neg = float(np.abs(np.sum(elev_diff[elev_diff < 0])))
        max_elev = float(np.max(elevations))
        min_elev = float(np.min(elevations))

        elev_series = elevations + [0]
        n = len(elev_series)

        def loc_last(prominence):
            peaks = find_peaks(elev_series, prominence=prominence)[0]
            return float(np.max(peaks) / n) if len(peaks) > 0 else 0.0

        last_points = min(500, len(elevations))
        elev_last = elevations[-last_points:]
        deniv_last_5km = float(np.sum(d for d in np.diff(elev_last) if d > 0))

        return {
            'distance_gpx_km':    round(dist, 2),
            'denivele_pos':       round(denivele_pos, 0),
            'denivele_neg':       round(denivele_neg, 0),
            'altitude_max':       round(max_elev, 0),
            'altitude_min':       round(min_elev, 0),
            'n_cols_hc':  get_number_peaks(elev_series, 800),
            'n_cols_cat1': get_number_peaks(elev_series, 640) - get_number_peaks(elev_series, 800),
            'n_cols_cat2': get_number_peaks(elev_series, 320) - get_number_peaks(elev_series, 640),
            'n_cols_cat3': get_number_peaks(elev_series, 160) - get_number_peaks(elev_series, 320),
            'n_cols_cat4': get_number_peaks(elev_series, 80)  - get_number_peaks(elev_series, 160),
            'loc_last_col_cat4':  round(loc_last(80), 4),
            'loc_last_col_cat3':  round(loc_last(160), 4),
            'loc_last_col_cat2':  round(loc_last(320), 4),
            'loc_last_col_cat1':  round(loc_last(640), 4),
            'loc_last_col_hc':    round(loc_last(800), 4),
            'gradient_last_1km':  gradient_last_km(lats, lons, elevations, 1),
            'gradient_last_3km':  gradient_last_km(lats, lons, elevations, 3),
            'gradient_last_5km':  gradient_last_km(lats, lons, elevations, 5),
            'denivele_last_5km':  round(deniv_last_5km, 0),
            'gradient_first_50km': gradient_first_km(lats, lons, elevations, 50),
            'denivele_first_50km': denivele_first_km(lats, lons, elevations, 50),
        }
    except Exception as e:
        print(f'  Erreur {os.path.basename(filepath)} : {e}')
        return {}

print('Fonctions chargees.')

Fonctions chargees.


In [15]:
# ── 1. Charger Komoot une fois ──────────────────────────────────────────────
df_komoot = pd.read_csv(KOMOOT_CSV)
print(f'Komoot : {len(df_komoot)} lignes')
print(f'Colonnes surface : {[c for c in df_komoot.columns if c not in ["fichier_gpx", "route_url"]]}')

Komoot : 7343 lignes
Colonnes surface : ['road_km', 'street_km', 'state_road_km', 'cycleway_km', 'off-grid_(unknown)_km', 'access_road_km', 'asphalt_km', 'paved_km', 'cobblestones_km', 'unknown_km', 'path_km', 'unpaved_km', 'singletrack_km', 'compacted_gravel_km', '_km', 'alpine_km', 'ferry_km', 'alpine_path_km']


In [20]:
# ── 2. Boucle par année : matching + Komoot + GPX ───────────────────────────

YEARS = range(2016, 2026)

for year in YEARS:
    sep = "=" * 50
    print(f"\n{sep}")
    print(f"  Traitement {year}")
    print(f"{sep}")

    # Charger matching
    path_match = os.path.join(MATCH_DIR, f"matching_{year}.csv")
    if not os.path.exists(path_match):
        print(f"  Fichier absent : {path_match}")
        continue

    df_match = pd.read_csv(path_match)
    df_match = df_match[df_match['statut'] == 'match']
    df_match = df_match.drop_duplicates(subset="fichier_gpx")
    print(f"  {year} : {len(df_match)} courses")

    # Merge Komoot
    df = df_match.merge(df_komoot, on="fichier_gpx", how="left")
    print(f"  Apres merge Komoot : {len(df)} lignes")
    print(f"  Sans Komoot : {df['route_url'].isna().sum()} courses")

    # Extraction GPX
    gpx_features = []
    for fname in tqdm(df["fichier_gpx"], desc=f"{year} GPX"):
        if pd.isna(fname):
            gpx_features.append({})
            continue
        path = find_gpx_path(str(fname), GPX_DIR)
        gpx_features.append(extract_gpx_features(path) if path else {})

    df_gpx = pd.DataFrame(gpx_features)

    # Assemblage
    df_final = pd.concat([df.reset_index(drop=True), df_gpx.reset_index(drop=True)], axis=1)
    cols_matching = ['jour', 'mois', 'annee', 'stage', 'nom_pcs', 'categorie', 'fichier_gpx']
    cols_komoot_c = [c for c in df_komoot.columns if c not in ['fichier_gpx']]
    cols_gpx      = list(df_gpx.columns)
    cols_finales  = [c for c in cols_matching + cols_komoot_c + cols_gpx if c in df_final.columns]
    df_final = df_final[cols_finales]

    # Sauvegarde par année
    out_path = os.path.join(OUTPUT_DIR, f"dataset_final_{year}.csv")
    df_final.to_csv(out_path, index=False)
    print(f"  Sauvegarde : {out_path}")
    print(f"  Shape : {df_final.shape}")
    print(f"  Courses avec Komoot : {df_final['route_url'].notna().sum()}")
    has_gpx = "distance_gpx_km" in df_final.columns
    print(f"  Courses avec GPX    : {df_final['distance_gpx_km'].notna().sum() if has_gpx else 'N/A'}")



  Traitement 2016
  Fichier absent : /Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all/matching_2016.csv

  Traitement 2017
  2017 : 595 courses
  Apres merge Komoot : 595 lignes
  Sans Komoot : 2 courses


2017 GPX:   0%|          | 0/595 [00:00<?, ?it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_35964/774333787.py:100: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last_5km = float(np.sum(d for d in np.diff(elev_last) if d > 0))
2017 GPX:   2%|▏         | 14/595 [00:01<01:22,  7.02it/s]


KeyboardInterrupt: 

In [21]:
# ── 3. Fusion de tous les fichiers par année ─────────────────────────────────

all_dfs = []
for year in YEARS:
    path = os.path.join(OUTPUT_DIR, f"dataset_final_{year}.csv")
    if os.path.exists(path):
        df_yr = pd.read_csv(path)
        all_dfs.append(df_yr)
        print(f"{year} : {len(df_yr)} courses")
    else:
        print(f"{year} : fichier absent")

df_all = pd.concat(all_dfs, ignore_index=True)
out_all = os.path.join(MATCH_DIR, "dataset_final_all.csv")
df_all.to_csv(out_all, index=False)
print(f"\nFusion sauvegardée : {out_all}")
print(f"Shape total : {df_all.shape}")
print(f"\nNaN par colonne :")
print(df_all.isna().sum()[df_all.isna().sum() > 0])


2016 : fichier absent
2017 : 595 courses
2018 : 1040 courses
2019 : 1062 courses
2020 : 448 courses
2021 : 700 courses
2022 : 813 courses
2023 : 937 courses
2024 : 890 courses
2025 : 906 courses

Fusion sauvegardée : /Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all/dataset_final_all.csv
Shape total : (7391, 47)

NaN par colonne :
route_url                  48
road_km                   282
street_km                1075
state_road_km             978
cycleway_km              3134
off-grid_(unknown)_km    3373
access_road_km           6282
asphalt_km                113
paved_km                  849
cobblestones_km          4489
unknown_km               2220
path_km                  3599
unpaved_km               4817
singletrack_km           5949
compacted_gravel_km      6797
_km                      7391
alpine_km                7376
ferry_km                 7389
alpine_path_km           7388
distance_gpx_km           161
denivele_pos              161
denivele_neg             

In [ ]:
# ── 4. Repêchage Komoot — Setup Selenium ────────────────────────────────────
# Lance cette cellule, connecte-toi à Komoot manuellement, puis passe à la suivante

import time, json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

CHECKPOINT = os.path.join(MATCH_DIR, "komoot_checkpoint_all.json")
KOMOOT_OUTPUT_CSV = os.path.join(MATCH_DIR, "komoot_surface_all.csv")

# Identifier les courses sans URL Komoot dans tous les fichiers par année
sans_url = []
for year in YEARS:
    path = os.path.join(OUTPUT_DIR, f"dataset_final_{year}.csv")
    if not os.path.exists(path):
        continue
    df_yr = pd.read_csv(path)
    manquants = df_yr[df_yr["route_url"].isna()]["fichier_gpx"].dropna().tolist()
    sans_url.extend(manquants)
    print(f"{year} : {len(manquants)} courses sans Komoot")

sans_url = list(dict.fromkeys(sans_url))  # dédoublonnage ordre préservé
print(f"\nTotal à scraper : {len(sans_url)} courses")

# Exclure ceux déjà dans le checkpoint avec un succès
already_done = set()
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        ckpt = json.load(f)
    already_done = {r["fichier_gpx"] for r in ckpt if r.get("route_url")}
    print(f"Déjà réussis dans checkpoint : {len(already_done)}")

todo_repechage = [f for f in sans_url if f not in already_done]
print(f"Restant à scraper : {len(todo_repechage)}")

# Lancer Chrome
options = Options()
options.add_argument("--window-size=1440,900")
options.add_argument("--use-gl=swiftshader")
options.add_argument("--enable-unsafe-swiftshader")
options.add_argument("--enable-webgl")
options.add_argument("--ignore-gpu-blocklist")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.get("https://www.komoot.com/login")
print("\nConnecte-toi à Komoot, puis lance la cellule suivante.")


2017 : 12 courses sans Komoot
2018 : 24 courses sans Komoot
2019 : 18 courses sans Komoot
2020 : 4 courses sans Komoot
2021 : 7 courses sans Komoot
2022 : 9 courses sans Komoot
2023 : 20 courses sans Komoot
2024 : 13 courses sans Komoot
2025 : 12 courses sans Komoot

Total à scraper : 119 courses
Déjà réussis dans checkpoint : 1771
Restant à scraper : 119

Connecte-toi à Komoot, puis lance la cellule suivante.


In [ ]:
# ── 5. Repêchage Komoot — Scraping + patch des fichiers finaux ──────────────

# Réutilise les fonctions find_gpx_path, upload_and_save, scrape_surface
# définies dans le scraper Komoot (à copier ici ou importer si dans un module)

def wait_for_text(text, timeout=180):
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: text in d.find_element(By.TAG_NAME, "body").text
        )
        return True
    except Exception:
        return False

def parse_km(text):
    text = text.replace("\u00a0", " ").strip()
    if "< 100" in text:
        return 0.0
    parts = text.split(" ")
    try:
        val = float(parts[0].replace(",", "."))
        unit = parts[1] if len(parts) > 1 else "km"
        return round(val / 1000 if unit == "m" else val, 3)
    except Exception:
        return None

def upload_and_save(filepath):
    driver.get("https://www.komoot.com/upload")
    time.sleep(2)
    try:
        file_input = WebDriverWait(driver, 180).until(
            EC.presence_of_element_located((By.XPATH, "//input[@type='file']"))
        )
        driver.execute_script("arguments[0].style.display = 'block';", file_input)
        file_input.send_keys(filepath)
    except Exception as e:
        print(f"  Upload introuvable : {e}")
        return None
    try:
        WebDriverWait(driver, 180).until(
            lambda d: "Import to Plan a Route" in d.find_element(By.TAG_NAME, "body").text
                      or "couldn't be imported" in d.find_element(By.TAG_NAME, "body").text
                      or "sorry" in d.find_element(By.TAG_NAME, "body").text.lower()
        )
    except Exception:
        print("  Timeout attente dialogue")
        return None
    body = driver.find_element(By.TAG_NAME, "body").text
    if "couldn't be imported" in body or "sorry" in body.lower():
        print("  Fichier rejeté par Komoot")
        return None
    if "Import to Plan a Route" not in body:
        print("  Import dialog non détecté")
        return None
    driver.execute_script("document.querySelectorAll('a[aria-label=\"Next\"]')[0]?.click();")
    if not wait_for_text("Road cycling", timeout=180):
        return None
    driver.execute_script("document.querySelectorAll('a[aria-label=\"Next\"]')[0]?.click();")
    try:
        WebDriverWait(driver, 30).until(
            lambda d: "Resolve Routing" in d.find_element(By.TAG_NAME, "body").text
                      or "Route matches to known ways" in d.find_element(By.TAG_NAME, "body").text
                      or "Route deviates from known ways" in d.find_element(By.TAG_NAME, "body").text
                      or "Route Saved" in d.find_element(By.TAG_NAME, "body").text
        )
    except Exception:
        return None
    body = driver.find_element(By.TAG_NAME, "body").text
    if any(x in body for x in ["Resolve Routing", "Route matches to known ways", "Route deviates from known ways"]):
        driver.execute_script("document.querySelectorAll('a[aria-label=\"Save\"]')[0]?.click();")
        if not wait_for_text("Route Saved", timeout=180):
            return None
    driver.execute_script("document.querySelectorAll('a.css-zi6wow')[0]?.click();")
    try:
        WebDriverWait(driver, 15).until(lambda d: "/tour/" in d.current_url)
    except Exception:
        return None
    time.sleep(1)
    return driver.current_url

def scrape_surface(route_url):
    driver.get(route_url)
    try:
        WebDriverWait(driver, 180).until(
            lambda d: len(d.find_elements(By.CSS_SELECTOR, "div.css-esdg3g")) > 0
        )
    except Exception:
        return {}
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
    time.sleep(1)
    entry = {}
    try:
        rows = driver.find_elements(By.CSS_SELECTOR, "div.css-esdg3g")
        for row in rows:
            try:
                label_el = row.find_element(By.CSS_SELECTOR, "button.css-1xu3avq")
                value_el = row.find_element(By.CSS_SELECTOR, "p.css-1wxtat5")
                label = label_el.text.strip().rstrip(":").lower().replace(" ", "_")
                if not label:
                    continue
                entry[f"{label}_km"] = parse_km(value_el.text)
            except Exception:
                continue
    except Exception as e:
        print(f"  scrape_surface : {e}")
    return entry

def save_checkpoint():
    tmp = CHECKPOINT + ".tmp"
    with open(tmp, "w") as f:
        json.dump(repechage_results, f, ensure_ascii=False, indent=2)
    os.replace(tmp, CHECKPOINT)

# Charger résultats existants du checkpoint
repechage_results = []
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        repechage_results = json.load(f)

# ── Scraping ─────────────────────────────────────────────────────────────────
for fname in tqdm(todo_repechage, desc="Repêchage Komoot"):
    filepath = find_gpx_path(fname, GPX_DIR)
    entry = {"fichier_gpx": fname}
    if not filepath:
        print(f"  Fichier GPX introuvable : {fname}")
        repechage_results.append(entry)
        save_checkpoint()
        continue
    route_url = None
    try:
        route_url = upload_and_save(filepath)
    except Exception as e:
        print(f"  Exception : {e}")
    if route_url:
        entry["route_url"] = route_url
        entry.update(scrape_surface(route_url))
        print(f"  OK : {fname}")
    else:
        print(f"  ECHEC : {fname}")
    repechage_results.append(entry)
    save_checkpoint()

driver.quit()
print("\nScraping terminé.")

# ── Patch des fichiers dataset_final_{year}.csv ──────────────────────────────
df_repechage = pd.DataFrame(repechage_results)
df_repechage = df_repechage[df_repechage["route_url"].notna()]  # seulement les succès
print(f"Nouvelles URLs récupérées : {len(df_repechage)}")

for year in YEARS:
    path = os.path.join(OUTPUT_DIR, f"dataset_final_{year}.csv")
    if not os.path.exists(path):
        continue
    df_yr = pd.read_csv(path)
    n_avant = df_yr["route_url"].isna().sum()
    if n_avant == 0:
        continue

    # Merge sur les lignes sans URL uniquement
    cols_komoot_new = [c for c in df_repechage.columns if c != "fichier_gpx"]
    df_yr = df_yr.merge(df_repechage[["fichier_gpx"] + cols_komoot_new],
                        on="fichier_gpx", how="left", suffixes=("", "_new"))

    # Patcher : pour chaque colonne Komoot, remplir le NaN avec la valeur _new
    for col in cols_komoot_new:
        col_new = col + "_new"
        if col_new in df_yr.columns:
            mask = df_yr[col].isna() & df_yr[col_new].notna()
            df_yr.loc[mask, col] = df_yr.loc[mask, col_new]
            df_yr.drop(columns=[col_new], inplace=True)

    n_apres = df_yr["route_url"].isna().sum()
    df_yr.to_csv(path, index=False)
    print(f"{year} : {n_avant - n_apres} courses patchées ({n_apres} encore sans URL)")

# Mettre à jour aussi komoot_surface_all.csv
df_komoot_existing = pd.read_csv(KOMOOT_OUTPUT_CSV)
df_komoot_updated = pd.concat([df_komoot_existing, df_repechage], ignore_index=True)
df_komoot_updated = df_komoot_updated.drop_duplicates(subset="fichier_gpx", keep="last")
df_komoot_updated.to_csv(KOMOOT_OUTPUT_CSV, index=False)
print(f"\nkomoot_surface_all.csv mis à jour : {len(df_komoot_updated)} lignes")

print("\nPatch terminé. Relance la cellule de fusion (cellule 5) pour régénérer dataset_final_all.csv")


Repêchage Komoot:   6%|▌         | 7/119 [00:00<00:03, 31.82it/s]

  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2017 Colombian Road National Championships.gpx
  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2017 Volta Ciclista a Catalunya Stage 7.gpx
  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2017 Tour of

Repêchage Komoot:  14%|█▍        | 17/119 [00:00<00:02, 38.10it/s]

  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2017 UCI Road World Championships.gpx
  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2017 Tour of Almaty Stage 2.gpx
  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2018 Colombian Road National Cham

Repêchage Komoot:  23%|██▎       | 27/119 [00:00<00:02, 42.60it/s]

...

  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2024 Grand Prix de la Ville d'Oran.gpx
  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2024 Grand Prix de la Ville d'Alger.gpx
  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2024 Tour of Estonia Sta

Repêchage Komoot:  99%|█████████▉| 118/119 [00:02<00:00, 50.15it/s]

  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2025 Tour du Limousin-Perigord - Nouvelle Aquitaine Stage 1.gpx
  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC : 2025 Lidl Deutschland Tour Prologue.gpx
  Exception : HTTPConnectionPool(host='localhost', port=52798): Max retries exceeded with url: /session/215ed6174eeb51243bda25c4a1203ab6/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=52798): Failed to establish a new connection: [Errno 61] Connection refused"))
  ECHEC :

Repêchage Komoot: 100%|██████████| 119/119 [00:02<00:00, 45.61it/s]



Scraping terminé.
Nouvelles URLs récupérées : 1837
2017 : 0 courses patchées (2 encore sans URL)
2018 : 0 courses patchées (11 encore sans URL)
2019 : 0 courses patchées (5 encore sans URL)
2020 : 0 courses patchées (4 encore sans URL)
2021 : 0 courses patchées (3 encore sans URL)
2022 : 0 courses patchées (4 encore sans URL)
2023 : 0 courses patchées (10 encore sans URL)
2024 : 0 courses patchées (8 encore sans URL)
2025 : 0 courses patchées (6 encore sans URL)

komoot_surface_all.csv mis à jour : 7338 lignes

Patch terminé. Relance la cellule de fusion (cellule 5) pour régénérer dataset_final_all.csv


In [22]:
# ── 3. Fusion de tous les fichiers par année ─────────────────────────────────

COLS_DROP = ['_km', 'alpine_km', 'ferry_km', 'alpine_path_km']

all_dfs = []
for year in YEARS:
    path = os.path.join(OUTPUT_DIR, f"dataset_final_{year}.csv")
    if os.path.exists(path):
        df_yr = pd.read_csv(path)
        all_dfs.append(df_yr)
        print(f"{year} : {len(df_yr)} courses")
    else:
        print(f"{year} : fichier absent")

df_all = pd.concat(all_dfs, ignore_index=True)

# Supprimer les colonnes inutiles
df_all.drop(columns=[c for c in COLS_DROP if c in df_all.columns], inplace=True)

# Remplacer les NaN par 0 sur les colonnes numériques
num_cols = df_all.select_dtypes(include='number').columns
df_all[num_cols] = df_all[num_cols].fillna(0)

out_all = os.path.join(MATCH_DIR, "dataset_final_all.csv")
df_all.to_csv(out_all, index=False)
print(f"\nFusion sauvegardée : {out_all}")
print(f"Shape total : {df_all.shape}")
print(f"\nNaN restants :")
print(df_all.isna().sum()[df_all.isna().sum() > 0])

2016 : fichier absent
2017 : 595 courses
2018 : 1040 courses
2019 : 1062 courses
2020 : 448 courses
2021 : 700 courses
2022 : 813 courses
2023 : 937 courses
2024 : 890 courses
2025 : 906 courses

Fusion sauvegardée : /Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all/dataset_final_all.csv
Shape total : (7391, 43)

NaN restants :
route_url    48
dtype: int64
